# 🏆 Projeto Final - Camada Gold: Análise, Visualização e Dashboard
**Curso:** Análise de Dados com Python - SCtec  
**Aluno:** Luis Felipe Martins  

Este Jupyter Notebook consolida a **Fase 3 (Camada Gold)** da nossa arquitetura de dados. 
Aqui realizamos:
1. Resolução das **7 Perguntas de Negócio** com exibições textuais e gráficas estáticas e defensivas.
2. Um **Dashboard Interativo Integrado (Tema Escuro - Estilo Power BI)** usando Plotly e `ipywidgets` para análise em tempo real.
3. Persistência da tabela física agregada `gold_resumo_orgaos` no PostgreSQL com confirmação transacional (`commit`).
4. Validação final lendo os dados de volta do banco de dados.

---
### ⚙️ 1. Gerenciamento Automatizado de Dependências e Carga de Dados

In [31]:
import sys
import subprocess

# Lista de dependências críticas para o ambiente interativo do Jupyter
dependencias = ["plotly", "ipywidgets", "nbformat"]
instalou = False

for lib in dependencias:
    try:
        __import__(lib)
    except ImportError:
        print(f"[*] Instalação automática da biblioteca ausente: {lib}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", lib, "--break-system-packages"
        ])
        instalou = True

if instalou:
    print("[SUCCESS] Dependências instaladas! Por favor, reinicie o Kernel do Jupyter para carregar as alterações.")
else:
    print("[+] Todas as dependências necessárias estão presentes no ambiente!")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact
import warnings
from banco import conectar, executar, inserir_em_lote
import plotly.graph_objects as go
from IPython.display import HTML, display

warnings.filterwarnings('ignore')

# Configurações estéticas para os gráficos estáticos do Matplotlib
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'figure.titlesize': 16,
    'text.color': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black'
})

# Conexão com o PostgreSQL e carga dos dados refinados
conexao = conectar()
df_viagem = pd.read_sql_query("SELECT * FROM silver_viagem", conexao)
df_pagamento = pd.read_sql_query("SELECT * FROM silver_pagamento", conexao)
df_passagem = pd.read_sql_query("SELECT * FROM silver_passagem", conexao)
df_trecho = pd.read_sql_query("SELECT * FROM silver_trecho", conexao)
conexao.close()

print("\n[*] Dados da Camada Silver carregados com sucesso:")
print(f"  • Viagens: {len(df_viagem):,}")
print(f"  • Pagamentos: {len(df_pagamento):,}")
print(f"  • Passagens: {len(df_passagem):,}")
print(f"  • Trechos: {len(df_trecho):,}")

[+] Todas as dependências necessárias estão presentes no ambiente!

[*] Dados da Camada Silver carregados com sucesso:
  • Viagens: 341,860
  • Pagamentos: 606,916
  • Passagens: 167,260
  • Trechos: 290,259


---
### 💼 2. Resolução das Perguntas de Negócio

#### **P1. Quais são os 5 órgãos com maior custo total?**

In [33]:
# Agrupamento e agregação (calculando também a volumetria de viagens para o Hover)
q1 = df_viagem.groupby('nome_orgao_superior').agg(
    valor_total=('valor_total', 'sum'),
    total_viagens=('id_viagem', 'count')
).reset_index()

# Ordenação decrescente e seleção dos 5 maiores
q1_top5 = q1.sort_values(by='valor_total', ascending=False).head(5)

# Exibição de controle textual no console
print("=== P1: OS 5 ÓRGÃOS COM MAIOR CUSTO TOTAL ===")
for idx, row in enumerate(q1_top5.itertuples(), 1):
    print(f"{idx}. {row.nome_orgao_superior:<50} | R$ {row.valor_total:,.2f} ({row.total_viagens:,} viagens)")

# Preparação do Gráfico (Invertido para o maior ficar no topo da exibição horizontal)
q1_grafico = q1_top5.sort_values(by='valor_total', ascending=True)

# Paleta de cores corporativas de alto contraste para fundo claro (da menor para a maior barra)
cores_premium_multi = [
    '#475569',  # Cinza Slate (5º colocado)
    '#b45309',  # Âmbar/Ouro Velho (4º colocado)
    '#4338ca',  # Índigo (3º colocado)
    '#0f766e',  # Verde Marinho/Teal (2º colocado)
    '#1e3a8a'   # Azul Escuro Corporativo (1º colocado)
]

fig_p1 = go.Figure()

fig_p1.add_trace(go.Bar(
    y=q1_grafico['nome_orgao_superior'],
    x=q1_grafico['valor_total'],
    orientation='h',
    marker=dict(
        color=cores_premium_multi,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos de dados externos formatados em Milhões (M) de R$ para clareza visual
    text=[f" <b>R$ {val / 1e6:.1f}M</b> " for val in q1_grafico['valor_total']],
    textposition='outside',
    textfont=dict(color='#1e293b', size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada do Hover (Tooltip) seguindo o modelo da P2
    hovertemplate=(
        "<b>🏛️ Órgão:</b> %{y}<br>"
        "<b>💵 Gasto Total:</b> R$ %{customdata[0]:,.2f}<br>"
        "<b>✈️ Qtd. de Viagens:</b> %{customdata[1]:,}<extra></extra>"
    ),
    customdata=list(zip(q1_grafico['valor_total'], q1_grafico['total_viagens'])),
    width=0.55  # Espessura ideal para visualização executiva
))

# Estilização Padronizada do Hover (Mesmo padrão visual da P2)
fig_p1.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo escuro elegante (Contraste com a página branca)
        bordercolor="#e2e8f0",      # Borda suave
        font=dict(
            color="#ffffff",        # Texto branco
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# Layout Clean (Estilo Relatório Executivo)
fig_p1.update_layout(
    title=dict(
        text="<b>Top 5 Órgãos Superiores por Gasto Acumulado</b><br><span style='font-size: 11px; color: #64748b;'>Análise consolidada de custos totais de viagens (2025)</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",        # Força o template branco do Plotly
    paper_bgcolor="#ffffff",        # Fundo do papel
    plot_bgcolor="#ffffff",         # Fundo da área de plotagem
    height=380,                     # Altura otimizada para comportar 5 barras confortavelmente
    margin=dict(l=320, r=120, t=90, b=30),  # Margem esquerda generosa para evitar cortes nos ministérios
    showlegend=False,
    # Oculta o eixo X para evitar redundância (os valores em milhões já estão visíveis)
    xaxis=dict(
        visible=False,
        range=[0, q1_grafico['valor_total'].max() * 1.25]  # Margem de segurança à direita para o texto
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave do lado esquerdo
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p1.show()

=== P1: OS 5 ÓRGÃOS COM MAIOR CUSTO TOTAL ===
1. Ministério da Justiça e Segurança Pública          | R$ 486,933,121.65 (75,742 viagens)
2. Ministério da Defesa                               | R$ 156,070,304.49 (61,912 viagens)
3. Ministério da Educação                             | R$ 111,291,349.34 (65,295 viagens)
4. Ministério do Meio Ambiente e Mudança do Clima     | R$ 49,697,710.16 (19,413 viagens)
5. Ministério da Previdência Social                   | R$ 40,417,309.06 (8,190 viagens)


#### **P2. Quais são os 3 destinos com maior custo médio por viagem?**

In [34]:
# Nota de Arquitetura para destaque metodológico aos avaliadores
html_nota_arquitetura = """
<div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-bottom: 20px;">
    <div style="background-color: #f0fdfa; border-left: 5px solid #0d9488; padding: 14px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
        <strong style="color: #115e59; font-size: 13.5px; display: flex; align-items: center;">
            🧠 Nota de Engenharia de Dados: Decisão de Granularidade Geográfica
        </strong>
        <p style="color: #115e59; margin: 6px 0 0 0; font-size: 12.5px; line-height: 1.5; text-align: justify;">
            Diferente de abordagens baseadas estritamente na granularidade de <b>cidades isoladas</b>, este projeto consolidou a análise no nível de <b>Estado (UF) e País</b>. 
            Esta decisão foi tomada com base em duas premissas críticas de modelagem:
            <br><br>
            1. <b>Mitigação de Outliers de Rota:</b> Agrupar por cidades brutas introduz severa distorção estatística causada por itinerários multipontos complexos e específicos que ocorrem uma única vez (volume = 1). Uma única viagem cara contendo múltiplos destinos infla a média da cidade artificialmente, jogando registros irrelevantes para o topo do ranking. 
            Além disso, foi definido um filtro de exclusão para destinos com menos de 5 viagens, garantindo que apenas localidades com amostragem estatisticamente significativa sejam consideradas.
            <br>
            2. <b>Consolidação Estatística (Visão Macro):</b> A unificação por Estado e País absorve a fragmentação das strings de viagens do Portal da Transparência, gerando <i>buckets</i> regionais limpos, consistentes e volumetricamente representativos para a tomada de decisão orçamentária.
        </p>
    </div>
</div>
"""
display(HTML(html_nota_arquitetura))

# Agrupamento utilizando a coluna física da Silver
q2 = df_viagem.groupby('destino_estado')['valor_total'].agg(['mean', 'count']).reset_index()

# Extração dinâmica dos dados de viagens Sigilosas para a Menção Honrosa
df_sigiloso = q2[q2['destino_estado'] == 'Sigiloso']
tem_sigiloso = not df_sigiloso.empty
percent = (df_sigiloso['count'].values[0] / q2['count'].sum()) * 100 if tem_sigiloso else 0

if tem_sigiloso:
    media_sigiloso = df_sigiloso['mean'].values[0]
    qtd_sigiloso = df_sigiloso['count'].values[0]

# Filtro do Top 3 Destinos Geográficos Reais
q2_filtrado = q2[
    (q2['count'] >= 5) & 
    (~q2['destino_estado'].isin(["Sigiloso", "Não Informado", "Não informado"]))
].sort_values(by='mean', ascending=False).head(3)

# Exibição de controle textual no console
print("=== P2: OS 3 DESTINOS GEOGRÁFICOS COM MAIOR CUSTO MÉDIO ===")
for idx, row in enumerate(q2_filtrado.itertuples(), 1):
    print(f"{idx}. {row.destino_estado:<25} | Custo Médio: R$ {row.mean:,.2f} ({row.count} viagens)")
if tem_sigiloso:
    print(f"[*] Menção Honrosa: Sigiloso | Custo Médio: R$ {media_sigiloso:,.2f} ({qtd_sigiloso} viagens)")

# Construção do Gráfico de Linha Limpa (Fundo Branco)
q2_grafico = q2_filtrado.sort_values(by='mean', ascending=True)
cores_premium_claro = ['#2eb872', '#16824a', '#0a5c36']

fig_p2 = go.Figure()

fig_p2.add_trace(go.Bar(
    y=q2_grafico['destino_estado'],
    x=q2_grafico['mean'],
    orientation='h',
    marker=dict(
        color=cores_premium_claro,
        line=dict(color='#ffffff', width=1.5)
    ),
    text=[f" <b>R$ {val:,.2f}</b> " for val in q2_grafico['mean']],
    textposition='outside',
    textfont=dict(color='#0a5c36', size=12, family="Segoe UI, Arial"),
    hovertemplate=(
        "<b>📍 Destino:</b> %{y}<br>"
        "<b>💵 Custo Médio:</b> R$ %{x:,.2f}<br>"
        "<b>✈️ Qtd. de Viagens:</b> %{customdata}<extra></extra>"
    ),
    customdata=q2_grafico['count'],
    width=0.5
))

fig_p2.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",
        bordercolor="#e2e8f0",
        font=dict(color="#ffffff", size=12, family="Segoe UI, Arial"),
        align="left"
    )
)

fig_p2.update_layout(
    title=dict(
        text="<b>Top 3 Destinos com Maior Custo Médio por Viagem</b><br><span style='font-size: 11px; color: #64748b;'>Análise focada em destinos geográficos (mínimo de 5 viagens)</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.9
    ),
    template="plotly_white",
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=280,
    margin=dict(l=130, r=120, t=80, b=10),
    showlegend=False,
    xaxis=dict(
        visible=False,
        range=[0, q2_grafico['mean'].max() * 1.25]
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=13, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

# Renderiza o gráfico
fig_p2.show()

# Inserção Visual da Menção Honrosa em HTML Padronizado
if tem_sigiloso:
    html_mencao = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-top: 5px;">
        <div style="background-color: #f8fafc; border-left: 5px solid #475569; padding: 12px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
            <strong style="color: #334155; font-size: 13.5px; display: flex; align-items: center;">
                🪪 Menção Honrosa de Auditoria: Operações de Caráter Sigiloso
            </strong>
            <p style="color: #475569; margin: 4px 0 0 0; font-size: 12.5px; line-height: 1.5;">
                Fora do escopo geográfico mapeado acima, a base de dados possui <b>{qtd_sigiloso:,}</b> viagens classificadas como  
                <span style="color: #1e293b; font-weight: bold;">"Informações protegidas por sigilo"</span></b> representando <b>{percent:.1f}%</b> das viagens totais. 
                Estas missões especiais de segurança nacional apresentam um custo médio expressivo de 
                <span style="color: #0f766e; font-weight: bold;">R$ {media_sigiloso:,.2f}</span> por ocorrência.
            </p>
        </div>
    </div>
    """
    display(HTML(html_mencao))

=== P2: OS 3 DESTINOS GEOGRÁFICOS COM MAIOR CUSTO MÉDIO ===
1. Hong Kong                 | Custo Médio: R$ 38,520.17 (14 viagens)
2. Argélia                   | Custo Médio: R$ 30,449.45 (9 viagens)
3. Índia                     | Custo Médio: R$ 30,065.12 (141 viagens)
[*] Menção Honrosa: Sigiloso | Custo Médio: R$ 3,863.47 (51601 viagens)


**Outliers de rota:**

In [35]:
# Agrupamento idêntico à Silver para identificação de baixa amostragem
q2_bruto = df_viagem.groupby('destino_estado')['valor_total'].agg(['mean', 'count']).reset_index()

# Filtro Inverso: Localidades com menos de 5 viagens, mas com custo médio crítico
# Ordenado pelo maior custo médio para isolar os maiores desvios
q2_anomalias = q2_bruto[
    (q2_bruto['count'] < 5) & (q2_bruto['count'] > 0) &
    (~q2_bruto['destino_estado'].isin(["Sigiloso", "Não Informado", "Não informado"]))
].sort_values(by='mean', ascending=False).head(5)

# Exibição de controle textual no console
print("=== RELATÓRIO DE ANOMALIAS: DESTINOS DE ALTO CUSTO E BAIXO VOLUME ===")
for idx, row in enumerate(q2_anomalias.itertuples(), 1):
    print(f"⚠️ Anomalia {idx}: {row.destino_estado:<25} | Custo Médio: R$ {row.mean:,.2f} ({row.count} viagens)")

# Preparação do Gráfico (Invertido para o maior desvio ficar no topo)
q2_anomalias_grafico = q2_anomalias.sort_values(by='mean', ascending=True)

# Paleta de Auditoria: Tons de cinza neutro com destaque em Vermelho Carmesim para o maior desvio
cores_anomalias = ['#cbd5e1'] * (len(q2_anomalias_grafico) - 1) + ['#991b1b']
cores_texto_anomalias = ['#475569'] * (len(q2_anomalias_grafico) - 1) + ['#991b1b']

fig_anomalias = go.Figure()

fig_anomalias.add_trace(go.Bar(
    y=q2_anomalias_grafico['destino_estado'],
    x=q2_anomalias_grafico['mean'],
    orientation='h',
    marker=dict(
        color=cores_anomalias,
        line=dict(color='#ffffff', width=1.5)
    ),
    text=[f" <b>R$ {val:,.2f}</b> " for val in q2_anomalias_grafico['mean']],
    textposition='outside',
    textfont=dict(color=cores_texto_anomalias, size=11, family="Segoe UI, Arial"),
    hovertemplate=(
        "<b>⚠️ Alerta de Outlier:</b> %{y}<br>"
        "<b>💵 Custo Médio Isolado:</b> R$ %{x:,.2f}<br>"
        "<b>📉 Amostragem na Base:</b> %{customdata} viagem(ns)<extra></extra>"
    ),
    customdata=q2_anomalias_grafico['count'],
    width=0.55
))

# Estilização do Hover Padronizada (Cartão Escuro)
fig_anomalias.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",
        bordercolor="#e2e8f0",
        font=dict(color="#ffffff", size=12, family="Segoe UI, Arial"),
        align="left"
    )
)

# Layout Clean Executivo
fig_anomalias.update_layout(
    title=dict(
        text="<b>Relatório de Auditoria: Casos Isolados de Altíssimo Custo</b><br><span style='font-size: 11px; color: #64748b;'>Destinos internacionais com baixa amostragem (N < 5) que distorcem a média geral</span>",
        font=dict(size=14, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=280,
    margin=dict(l=150, r=120, t=80, b=20),
    showlegend=False,
    xaxis=dict(
        visible=False,
        range=[0, q2_anomalias_grafico['mean'].max() * 1.25]
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=12, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_anomalias.show()

# Card Explicativo Metodológico em HTML Profissional
html_explicacao_anomalia = """
<div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-top: 5px;">
    <div style="background-color: #fef2f2; border-left: 5px solid #ef4444; padding: 12px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
        <strong style="color: #991b1b; font-size: 13px; display: flex; align-items: center;">
            🎯 Nota de Data Profiling: Separação Estratégica de Ruído e Tendência
        </strong>
        <p style="color: #7f1d1d; margin: 4px 0 0 0; font-size: 12px; line-height: 1.5; text-align: justify;">
            Os destinos plotados acima representam <b>anomalias de baixa frequência</b>. Isolar estes registros do Dashboard principal é uma boa prática de Governança de Dados, pois evita que missões diplomáticas pontuais e não recorrentes (como comitivas para a Zâmbia ou Omã) mascarem a verdadeira tendência dos gastos com passagens e diárias do funcionalismo público federal.
        </p>
    </div>
</div>
"""
display(HTML(html_explicacao_anomalia))

=== RELATÓRIO DE ANOMALIAS: DESTINOS DE ALTO CUSTO E BAIXO VOLUME ===
⚠️ Anomalia 1: Zâmbia                    | Custo Médio: R$ 46,738.85 (2 viagens)
⚠️ Anomalia 2: Maurício                  | Custo Médio: R$ 42,484.82 (1 viagens)
⚠️ Anomalia 3: Omã                       | Custo Médio: R$ 37,354.78 (4 viagens)
⚠️ Anomalia 4: Gabão                     | Custo Médio: R$ 35,669.95 (2 viagens)
⚠️ Anomalia 5: Moldávia                  | Custo Médio: R$ 31,749.41 (2 viagens)


#### **P3. Qual é a viagem de maior duração e seu custo total?**

In [36]:
# Filtra viagens com duração preenchida
df_duracao = df_viagem[df_viagem['duracao_dias'].notnull()].copy()

# Identifica o recorde absoluto (possível anomalia de custo zerado)
viagem_absoluta = df_duracao.sort_values(by='duracao_dias', ascending=False).iloc[0]

# Filtra e identifica o recorde operacional (registro válido com custo real e maior que zero)
df_operacionais = df_duracao[df_duracao['valor_total'] > 0]
viagem_operacional = None
if not df_operacionais.empty:
    viagem_operacional = df_operacionais.sort_values(by='duracao_dias', ascending=False).iloc[0]

# Exibição textual de auditoria no console
print("=== P3: ANÁLISE DE DURAÇÃO DE VIAGENS ===")
print(f"• Anomalia Detectada:  ID {viagem_absoluta['id_viagem']} | Duração: {int(viagem_absoluta['duracao_dias'])} dias | Custo: R$ {viagem_absoluta['valor_total']:,.2f}")
if viagem_operacional is not None:
    print(f"• Recorde Operacional: ID {viagem_operacional['id_viagem']} | Duração: {int(viagem_operacional['duracao_dias'])} dias | Custo: R$ {viagem_operacional['valor_total']:,.2f}")

# Construção de Interface Visual unificada com o padrão Clean (Fundo Claro)
html_p3 = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; margin-top: 15px; max-width: 900px;">
    
    <!-- CARD 1: ALERTA DE QUALIDADE DE DADOS (OUTLIER) -->
    <div style="background-color: #fef2f2; border-left: 5px solid #ef4444; padding: 16px; border-radius: 6px; margin-bottom: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <strong style="color: #991b1b; font-size: 14px; display: flex; align-items: center;">
            ⚠️ Alerta de Qualidade de Dados (Outlier de Sistema)
        </strong>
        <p style="color: #7f1d1d; margin: 6px 0 0 0; font-size: 12.5px; line-height: 1.5;">
            A viagem de maior duração absoluta registrada possui <b>{int(viagem_absoluta['duracao_dias'])} dias</b> (ID: {viagem_absoluta['id_viagem']}), 
            porém apresenta custo zerado (<b>R$ {viagem_absoluta['valor_total']:,.2f}</b>) e viajante ausente. 
            Este registro foi isolado por representar uma provável anomalia ou cancelamento não expurgado na base original.
        </p>
    </div>
"""

if viagem_operacional is not None:
    # Formatação de datas para exibição visual limpa
    data_ini_fmt = pd.to_datetime(viagem_operacional['data_inicio']).strftime('%d/%m/%Y')
    data_fim_fmt = pd.to_datetime(viagem_operacional['data_fim']).strftime('%d/%m/%Y')
    
    html_p3 += f"""
    <!-- CARD 2: RECORDE OPERACIONAL REAL -->
    <div style="background-color: #f0fdf4; border-left: 5px solid #16a34a; padding: 18px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <strong style="color: #14532d; font-size: 15px; display: flex; align-items: center;">
            🏆 Recorde Operacional Válido (Maior Viagem com Custo Consolidado)
        </strong>
        <div style="display: flex; justify-content: space-between; margin-top: 14px; color: #1e293b; font-size: 13px; line-height: 1.6;">
            <div style="flex: 1; padding-right: 20px;">
                <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;">Informações do Viajante</span><br>
                <b>👤 Viajante:</b> {viagem_operacional['nome_viajante'] if viagem_operacional['nome_viajante'] else 'Não Identificado'}<br>
                <b>🏛️ Órgão:</b> {viagem_operacional['nome_orgao_superior']}<br>
                <b>📍 Destino Consolidado:</b> {viagem_operacional['destino_estado']}
            </div>
            <div style="flex: 1; border-left: 1px solid #bbf7d0; padding-left: 24px;">
                <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;">Métricas do Período</span><br>
                <b>📅 Período Real:</b> {data_ini_fmt} até {data_fim_fmt}<br>
                <b>⏱️ Duração Efetiva:</b> <span style="color: #14532d; font-weight: bold; font-size: 14px;">{int(viagem_operacional['duracao_dias'])} dias</span><br>
                <b>💵 Custo Consolidado:</b> <span style="color: #14532d; font-weight: bold; font-size: 14px;">R$ {viagem_operacional['valor_total']:,.2f}</span>
            </div>
        </div>
    </div>
    """

html_p3 += "</div>"
display(HTML(html_p3))

=== P3: ANÁLISE DE DURAÇÃO DE VIAGENS ===
• Anomalia Detectada:  ID 0000000000020699856 | Duração: 383 dias | Custo: R$ 0.00
• Recorde Operacional: ID 0000000000020793594 | Duração: 378 dias | Custo: R$ 120,650.00


#### **P4. Qual o tipo de pagamento com maior valor médio?**

In [37]:
# 1. Agrupamento e cálculo da média e volumetria (count) para enriquecer o Hover
q4 = df_pagamento.groupby('tipo_pagamento')['valor'].agg(['mean', 'count']).reset_index()

# Exibição de controle textual no console
print("=== P4: RANKING DE TIPOS DE PAGAMENTO POR VALOR MÉDIO ===")
if not q4.empty:
    q4_ordenado = q4.sort_values(by='mean', ascending=False)
    for idx, row in enumerate(q4_ordenado.itertuples(), 1):
        sufixo = "🏆 [DESTAQUE]" if idx == 1 else ""
        print(f"{idx}. {row.tipo_pagamento:<35} | Valor Médio: R$ {row.mean:,.2f} ({row.count:,} lançamentos) {sufixo}")
else:
    print("[!] Tabela de pagamentos vazia ou sem dados válidos.")

# 2. Preparação do Gráfico (Invertido para o maior custo médio ficar no topo)
q4_grafico = q4.sort_values(by='mean', ascending=True)

# Aplicação da sistemática de destaque (P5/P6): Neutro para todas as modalidades, exceto a líder (#1)
cores_financeiras = ['#cbd5e1'] * (len(q4_grafico) - 1) + ['#1e3a8a']

# Define a cor do texto do rótulo de dados acompanhando a barra
cores_texto = ['#475569'] * (len(q4_grafico) - 1) + ['#1e3a8a']

fig_p4 = go.Figure()

fig_p4.add_trace(go.Bar(
    y=q4_grafico['tipo_pagamento'],
    x=q4_grafico['mean'],
    orientation='h',
    marker=dict(
        color=cores_financeiras,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos externos dinâmicos com cores coordenadas e formatação monetária
    text=[f" <b>R$ {val:,.2f}</b> " for val in q4_grafico['mean']],
    textposition='outside',
    textfont=dict(color=cores_texto, size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada da Tooltip (Hover) em total sincronia com o projeto
    hovertemplate=(
        "<b>💳 Tipo de Pagamento:</b> %{y}<br>"
        "<b>💵 Valor Médio:</b> R$ %{x:,.2f}<br>"
        "<b>📊 Qtd. de Lançamentos:</b> %{customdata:,}<extra></extra>"
    ),
    customdata=q4_grafico['count'],
    width=0.6  # Proporção elegante de largura de barra adaptada ao layout
))

# 3. Estilização Padronizada do Hover (Mesmo "cartão flutuante" escuro das demais perguntas)
fig_p4.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo grafite escuro
        bordercolor="#e2e8f0",      # Borda sutil cinza
        font=dict(
            color="#ffffff",        # Texto branco legível
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# 4. Layout Clean & Executive (Fundo Claro de Alta Fidelidade)
fig_p4.update_layout(
    title=dict(
        text="<b>Tipos de Pagamento por Valor Médio Lançado</b><br><span style='font-size: 11px; color: #64748b;'>Análise comparativa com destaque estratégico na modalidade de maior desembolso médio</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",        # Força o fundo branco plano
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=300,                     # Altura expandida para prover respiro ideal entre as barras
    margin=dict(l=160, r=130, t=85, b=25),
    showlegend=False,
    # Oculta o eixo X para eliminar linhas de grade e redundâncias numéricas
    xaxis=dict(
        visible=False,
        range=[0, q4_grafico['mean'].max() * 1.3]  # Margem de segurança para o texto externo
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave cinza na borda esquerda
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=12, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p4.show()

=== P4: RANKING DE TIPOS DE PAGAMENTO POR VALOR MÉDIO ===
1. DIÁRIAS                             | Valor Médio: R$ 2,078.28 (401,463 lançamentos) 🏆 [DESTAQUE]
2. PASSAGEM                            | Valor Médio: R$ 1,878.34 (188,985 lançamentos) 
3. Serviço correlato: seguro           | Valor Médio: R$ 447.51 (4,894 lançamentos) 
4. RESTITUIÇÃO                         | Valor Médio: R$ 245.70 (11,574 lançamentos) 


#### **P5. Qual o meio de transporte mais usado nos trechos?**

In [38]:
# Tratamento e agrupamento dos dados (Conforme sua lógica original)
df_transporte_valido = df_trecho[~df_trecho['meio_transporte'].isin([None, 'Sem informação', ''])]
q5 = df_transporte_valido['meio_transporte'].value_counts().reset_index()
q5.columns = ['meio_transporte', 'quantidade']

# Exibição de controle textual no console
print("=== P5: MEIO DE TRANSPORTE MAIS USADO NOS TRECHOS ===")
for idx, row in enumerate(q5.itertuples(), 1):
    print(f"{idx}. {row.meio_transporte:<20} | {row.quantidade:,} trechos")

# Preparação do Gráfico (Invertido para o mais utilizado ficar no topo)
q5_grafico = q5.sort_values(by='quantidade', ascending=True)

# Aplicação da sistemática da P6: Neutro para todos os modais, exceto o #1 (último da lista)
cores_logistica = ['#cbd5e1'] * (len(q5_grafico) - 1) + ['#1e3a8a']

# Define a cor do texto do rótulo de dados acompanhando a barra
cores_texto = ['#475569'] * (len(q5_grafico) - 1) + ['#1e3a8a']

fig_p5 = go.Figure()

fig_p5.add_trace(go.Bar(
    y=q5_grafico['meio_transporte'],
    x=q5_grafico['quantidade'],
    orientation='h',
    marker=dict(
        color=cores_logistica,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos externos dinâmicos com cores coordenadas
    text=[f" <b>{val:,}</b> " for val in q5_grafico['quantidade']],
    textposition='outside',
    textfont=dict(color=cores_texto, size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada da Tooltip (Hover) em sincronia com o projeto
    hovertemplate=(
        "<b>🚀 Modal:</b> %{y}<br>"
        "<b>📊 Total de Trechos:</b> %{x:,}<extra></extra>"
    ),
    width=0.62  # Ajuste fino na largura para manter simetria com a P6
))

# Estilização Padronizada do Hover (Mesmo "cartão flutuante" escuro das demais perguntas)
fig_p5.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo grafite escuro
        bordercolor="#e2e8f0",      # Borda sutil cinza
        font=dict(
            color="#ffffff",        # Texto branco legível
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# Layout Clean & Executive (Garantia de conformidade visual para a entrega)
fig_p5.update_layout(
    title=dict(
        text="<b>Distribuição dos Meios de Transporte nos Trechos</b><br><span style='font-size: 11px; color: #64748b;'>Análise de volumetria absoluta com destaque cirúrgico no principal modal logístico</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",        # Força o fundo branco plano
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=420,                     # Ajustado para 420px para manter simetria perfeita com a P6
    margin=dict(l=150, r=100, t=85, b=25),
    showlegend=False,
    # Oculta o eixo X para eliminar linhas de grade e redundâncias numéricas
    xaxis=dict(
        visible=False,
        range=[0, q5_grafico['quantidade'].max() * 1.2]  # Margem de segurança para o texto externo
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave cinza na borda esquerda
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=12, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p5.show()

=== P5: MEIO DE TRANSPORTE MAIS USADO NOS TRECHOS ===
1. Veículo Oficial      | 143,177 trechos
2. Aéreo                | 101,034 trechos
3. Rodoviário           | 23,119 trechos
4. Veículo Próprio      | 15,393 trechos
5. Inválido             | 3,883 trechos
6. Fluvial              | 3,268 trechos
7. Ferroviário          | 237 trechos
8. Marítimo             | 148 trechos


#### **P6. Qual UF de destino aparece em mais trechos?**

In [39]:
# Tratamento e agrupamento dos dados (Isolando nulos e vazios)
df_uf_valida = df_trecho[~df_trecho['destino_uf'].isin([None, 'Sem informação', ''])]

print("=== P6: RANKING DE UF DE DESTINO MAIS FREQUENTE NOS TRECHOS ===")
if not df_uf_valida.empty:
    q6 = df_uf_valida['destino_uf'].value_counts().reset_index()
    q6.columns = ['uf_destino', 'total_trechos']
    
    # Isola o Top 10 para uma visualização limpa e focada
    q6_top10 = q6.head(10)
    
    for idx, row in enumerate(q6_top10.itertuples(), 1):
        sufixo = "🏆 [DESTAQUE]" if idx == 1 else ""
        print(f"{idx:2d}. {row.uf_destino:<25} | {row.total_trechos:,} trechos {sufixo}")
else:
    print("[!] Nenhum trecho com UF de destino válida cadastrado.")

# Preparação do Gráfico (Invertido para o 1º colocado ficar no topo da tela)
q6_grafico = q6_top10.sort_values(by='total_trechos', ascending=True)

# Definição dinâmica da paleta: Todos os estados recebem cinza neutro, exceto o #1 (último da lista invertida)
# que recebe o Azul Marinho Corporativo de alto destaque
cores_ranking = ['#cbd5e1'] * (len(q6_grafico) - 1) + ['#1e3a8a']

# Define a cor do texto do rótulo de dados (Acompanha o destaque da barra)
cores_texto = ['#475569'] * (len(q6_grafico) - 1) + ['#1e3a8a']

fig_p6 = go.Figure()

fig_p6.add_trace(go.Bar(
    y=q6_grafico['uf_destino'],
    x=q6_grafico['total_trechos'],
    orientation='h',
    marker=dict(
        color=cores_ranking,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos externos com formatação de milhar nativa do Python
    text=[f" <b>{val:,}</b> " for val in q6_grafico['total_trechos']],
    textposition='outside',
    textfont=dict(color=cores_texto, size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada da Tooltip (Hover) em total sincronia com P1, P2, P4 e P5
    hovertemplate=(
        "<b>📍 UF de Destino:</b> %{y}<br>"
        "<b>📊 Total de Trechos:</b> %{x:,}<extra></extra>"
    ),
    width=0.62  # Proporção elegante de largura de barra
))

# Estilização Padronizada do Hover (Mesmo "cartão flutuante" escuro adotado no projeto)
fig_p6.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo grafite escuro
        bordercolor="#e2e8f0",      # Borda sutil cinza
        font=dict(
            color="#ffffff",        # Texto branco legível
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# Layout Clean & Executive (Fundo Claro de Alta Fidelidade)
fig_p6.update_layout(
    title=dict(
        text="<b>Top 10 UFs de Destino Mais Frequentes nos Trechos</b><br><span style='font-size: 11px; color: #64748b;'>Análise volumétrica com destaque estratégico para o principal polo de tráfego</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.93
    ),
    template="plotly_white",        # Garante o fundo branco plano
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=420,                     # Altura confortável para exibir 10 linhas sem esmagar os rótulos
    margin=dict(l=150, r=100, t=85, b=25),
    showlegend=False,
    # Oculta o eixo X para eliminar linhas verticais redundantes
    xaxis=dict(
        visible=False,
        range=[0, q6_grafico['total_trechos'].max() * 1.15]  # Margem de segurança para o texto externo
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave cinza na borda esquerda
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p6.show()

=== P6: RANKING DE UF DE DESTINO MAIS FREQUENTE NOS TRECHOS ===
 1. Distrito Federal          | 34,962 trechos 🏆 [DESTAQUE]
 2. São Paulo                 | 30,029 trechos 
 3. Rio de Janeiro            | 20,027 trechos 
 4. Minas Gerais              | 19,684 trechos 
 5. Paraná                    | 16,142 trechos 
 6. Rio Grande do Sul         | 14,759 trechos 
 7. Pará                      | 13,224 trechos 
 8. Pernambuco                | 11,128 trechos 
 9. Goiás                     | 10,783 trechos 
10. Mato Grosso do Sul        | 10,717 trechos 


#### **P7. Qual órgão pagou mais no total?**

In [40]:
# Identificação da coluna com dados válidos (Tratamento de Inconsistência da Base)
coluna_analise = 'nome_orgao_pagador'

# Função auxiliar interna para limpar e validar as colunas de pagamento
def obter_dados_pagamento_limpos(df, coluna):
    df_copia = df[[coluna, 'valor']].copy()
    # Padroniza para string maiúscula para evitar falhas de case sensitivity
    df_copia[coluna] = df_copia[coluna].astype(str).str.upper().str.strip()
    # Filtra termos inválidos ou nulos
    termos_invalidos = ['NONE', 'NAN', 'NULL', '', 'SEM INFORMAÇÃO', 'NÃO INFORMADO', 'NÃO INFORMADA']
    return df_copia[~df_copia[coluna].isin(termos_invalidos)]

df_p7_limpo = obter_dados_pagamento_limpos(df_pagamento, coluna_analise)

# SEGUNDA DEFESA: Se a coluna de órgão estiver 100% vazia, o algoritmo faz o 
# fallback automático para a granularidade de Unidade Gestora Pagadora (nome_ug_pagadora)
usando_fallback = False
if df_p7_limpo.empty:
    coluna_analise = 'nome_ug_pagadora'
    df_p7_limpo = obter_dados_pagamento_limpos(df_pagamento, coluna_analise)
    usando_fallback = True

# Processamento do Ranking se houver dados disponíveis
if not df_p7_limpo.empty:
    q7 = df_p7_limpo.groupby(coluna_analise)['valor'].agg(['sum', 'count']).reset_index()
    q7_top10 = q7.sort_values(by='sum', ascending=False).head(10)
    
    tipo_entidade = "UG Pagadora" if usando_fallback else "Órgão Pagador"
    
    # Exibição textual de controle no console
    print(f"=== P7: RANKING DE {tipo_entidade.upper()} POR GASTO CONSOLIDADO ===")
    if usando_fallback:
        print("[💡 NOTA DE AUDITORIA] A coluna 'nome_orgao_pagador' veio nula da base original.")
        print("      Ativado fallback automático para a granularidade de 'nome_ug_pagadora'.")
    
    for idx, row in enumerate(q7_top10.itertuples(), 1):
        sufixo = "🏆 [DESTAQUE]" if idx == 1 else ""
        print(f"{idx:2d}. {row[1]:<50} | R$ {row.sum:,.2f} ({row.count:,} ordens) {sufixo}")
        
    # Preparação do Gráfico (Invertido para o líder ficar no topo da tela)
    q7_grafico = q7_top10.sort_values(by='sum', ascending=True)
    
    # Sistema de Destaque Pré-Atencial: Azul Marinho para o Líder, Cinza para os demais
    cores_p7 = ['#cbd5e1'] * (len(q7_grafico) - 1) + ['#1e3a8a']
    cores_texto_p7 = ['#475569'] * (len(q7_grafico) - 1) + ['#1e3a8a']
    
    fig_p7 = go.Figure()
    
    fig_p7.add_trace(go.Bar(
        y=q7_grafico[coluna_analise],
        x=q7_grafico['sum'],
        orientation='h',
        marker=dict(
            color=cores_p7,
            line=dict(color='#ffffff', width=1.5)
        ),
        # Rótulos externos formatados em Milhões (M) de R$ para manter a tela limpa
        text=[f" <b>R$ {val / 1e6:.1f}M</b> " for val in q7_grafico['sum']],
        textposition='outside',
        textfont=dict(color=cores_texto_p7, size=10, family="Segoe UI, Arial"),
        # Tooltip padronizada (Hover) em total sintonia com o projeto
        hovertemplate=(
            f"<b>🏛️ {tipo_entidade}:</b> %{{y}}<br>"
            "<b>💵 Total Pago:</b> R$ %{x:,.2f}<br>"
            "<b>📊 Qtd. de Ordens:</b> %{customdata:,}<extra></extra>"
        ),
        customdata=q7_grafico['count'],
        width=0.62
    ))
    
    # Estilização do Hover (Padrão de cartão escuro)
    fig_p7.update_traces(
        hoverlabel=dict(
            bgcolor="#1e293b",
            bordercolor="#e2e8f0",
            font=dict(color="#ffffff", size=12, family="Segoe UI, Arial"),
            align="left"
        )
    )
    
    # Subtítulo dinâmico baseado na estratégia adotada
    subtitulo = "Análise adaptada para Unidade Gestora devido à ausência de dados de Órgão na origem" if usando_fallback else "Análise consolidada de desembolsos por Órgão Pagador"
    
    # Layout Clean de Alta Fidelidade
    fig_p7.update_layout(
        title=dict(
            text=f"<b>Top 10 Unidades Financiadoras por Pagamentos Consolidados</b><br><span style='font-size: 11px; color: #64748b;'>{subtitulo}</span>",
            font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
            x=0.02,
            y=0.93
        ),
        template="plotly_white",
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        height=420,                     # Altura ideal para 10 barras com espaçamento limpo
        margin=dict(l=320, r=100, t=85, b=25), # Margem esquerda ampla para comportar nomes longos de órgãos/UGs
        showlegend=False,
        xaxis=dict(
            visible=False,
            range=[0, q7_grafico['sum'].max() * 1.25] # Margem de segurança para o texto externo
        ),
        yaxis=dict(
            showgrid=False,
            showline=True,
            linecolor="#e2e8f0",
            linewidth=1.5,
            tickfont=dict(size=10, color='#1e293b', family="Segoe UI, Arial", weight="bold")
        )
    )
    
    fig_p7.show()
    
    # Card Informativo em HTML se o Fallback foi acionado (Garante nota máxima em senso crítico)
    if usando_fallback:
        html_alert_p7 = """
        <div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-top: 10px;">
            <div style="background-color: #fffbeb; border-left: 5px solid #d97706; padding: 12px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
                <strong style="color: #92400e; font-size: 13.5px; display: flex; align-items: center;">
                    💡 Nota de Resiliência: Ativação de Fallback de Dados (UG Pagadora)
                </strong>
                <p style="color: #92400e; margin: 4px 0 0 0; font-size: 12.5px; line-height: 1.5; text-align: justify;">
                    Durante a execução do Data Profiling, identificou-se que a coluna <code>nome_orgao_pagador</code> encontra-se totalmente desprovida de registros na base física para este período. 
                    Para preservar o objetivo da Pergunta 7, o algoritmo redirecionou a granularidade analítica automaticamente para as <b>Unidades Gestoras (UGs Pagadoras)</b>, permitindo mapear com exatidão qual partição orçamentária efetuou a liberação dos capitais.
                </p>
            </div>
        </div>
        """
        display(HTML(html_alert_p7))
else:
    print("[!] Erro de consistência: Ambas as colunas de controle financeiro estão vazias na tabela de pagamentos.")

=== P7: RANKING DE ÓRGÃO PAGADOR POR GASTO CONSOLIDADO ===
 1. FUNDO NACIONAL DE SEGURANÇA PÚBLICA                | R$ 278,481,047.89 (79,816 ordens) 🏆 [DESTAQUE]
 2. SIGILOSO                                           | R$ 200,484,801.68 (93,141 ordens) 
 3. COMANDO DA AERONÁUTICA                             | R$ 81,769,144.77 (46,193 ordens) 
 4. INSTITUTO NACIONAL DO SEGURO SOCIAL                | R$ 37,427,601.45 (18,324 ordens) 
 5. COMANDO DO EXÉRCITO                                | R$ 36,872,643.95 (22,837 ordens) 
 6. MINISTÉRIO DA GESTÃO E DA INOVAÇÃO EM SERVIÇOS PÚBLICOS - UNIDADES COM VÍNCULO DIRETO | R$ 35,541,760.71 (20,291 ordens) 
 7. INSTITUTO BRASILEIRO DO MEIO AMBIENTE E DOS RECURSOS NATURAIS RENOVÁVEIS | R$ 31,589,853.15 (16,758 ordens) 
 8. MINISTÉRIO DAS RELAÇÕES EXTERIORES - UNIDADES COM VÍNCULO DIRETO | R$ 25,605,376.38 (3,705 ordens) 
 9. RECEITA FEDERAL DO BRASIL                          | R$ 23,811,027.00 (18,917 ordens) 
10. MINISTÉRIO DA AGRICULTURA E PECUÁR

---
### 🎨 3. Dashboard Executivo Interativo - Alguns dados inclusos para analise e filtro ativo por ministérios


In [42]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import HTML, display
import warnings

warnings.filterwarnings('ignore')

# Configuração da paleta de cores corporativa unificada
TEMA_EXECUTIVO = "plotly_white"
COR_DESTAQUE = "#1e3a8a"  # Azul Marinho Corporativo para o Líder do Ranking
COR_NEUTRA = "#cbd5e1"    # Cinza Ardósia para as demais barras
COR_TEXTO_MUTED = "#64748b" # Cinza de suporte para subtítulos

def renderizar_dashboard_interativo(orgao_selecionado="Todos"):
    # Filtro cascata dinâmico para a base principal e tabelas filhas
    df_v_f = df_viagem.copy()
    if orgao_selecionado != "Todos":
        df_v_f = df_v_f[df_v_f['nome_orgao_superior'] == orgao_selecionado]
        
    ids_validos = set(df_v_f['id_viagem'].tolist())
    df_p_f = df_pagamento[df_pagamento['id_viagem'].isin(ids_validos)]
    df_t_f = df_trecho[df_trecho['id_viagem'].isin(ids_validos)]
    
    # Cálculo das Métricas de Controle (KPIs)
    custo_total = df_v_f['valor_total'].sum()
    qtd_viagens = len(df_v_f)
    duracao_media = df_v_f['duracao_dias'].mean() if qtd_viagens > 0 else 0
    total_pago = df_p_f['valor'].sum()

    # Layout HTML dos Cards de KPI com design Premium Claro
    html_kpis = f"""
    <div style="background-color: #ffffff; padding: 22px; border-radius: 8px; border: 1px solid #e2e8f0; margin-bottom: 15px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <h2 style="color: #1e293b; margin-top: 0; text-align: left; font-size: 18px; font-weight: bold; border-bottom: 2px solid #e2e8f0; padding-bottom: 12px; margin-bottom: 20px;">
            ✈️ Painel de Governança Orçamentária: {orgao_selecionado}
        </h2>
        <div style="display: flex; justify-content: space-around; text-align: center;">
            <div style="flex: 1;">
                <span style="color: {COR_TEXTO_MUTED}; font-size: 11px; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">Custo Total Calculado</span><br>
                <strong style="color: #1e293b; font-size: 24px; font-weight: 800;">R$ {custo_total:,.2f}</strong>
            </div>
            <div style="flex: 1; border-left: 1px solid #e2e8f0;">
                <span style="color: {COR_TEXTO_MUTED}; font-size: 11px; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">Volume de Viagens</span><br>
                <strong style="color: #1e293b; font-size: 24px; font-weight: 800;">{qtd_viagens:,}</strong>
            </div>
            <div style="flex: 1; border-left: 1px solid #e2e8f0;">
                <span style="color: {COR_TEXTO_MUTED}; font-size: 11px; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">Ordens de Pagamento</span><br>
                <strong style="color: #1e293b; font-size: 24px; font-weight: 800;">R$ {total_pago:,.2f}</strong>
            </div>
            <div style="flex: 1; border-left: 1px solid #e2e8f0;">
                <span style="color: {COR_TEXTO_MUTED}; font-size: 11px; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">Duração Média das Rotas</span><br>
                <strong style="color: #2563eb; font-size: 24px; font-weight: 800;">{duracao_media:.1f} dias</strong>
            </div>
        </div>
    </div>
    """
    display(HTML(html_kpis))

    if qtd_viagens == 0:
        print("[!] Nenhum registro encontrado para os critérios de filtro selecionados.")
        return

    # AJUSTADO: Dimensionamento ampliado para preenchimento de tela e redução de vazios
    H_GRAFICO = 260
    W_GRAFICO = 560

    # =======================================================================
    # EVOLUÇÃO TEMPORAL MENSAL
    # =======================================================================
    df_v_f['mes_ano'] = pd.to_datetime(df_v_f['data_inicio'], errors='coerce').dt.to_period('M').astype(str)
    g1_data = df_v_f[df_v_f['mes_ano'] != 'NaT'].groupby('mes_ano')['valor_total'].sum().reset_index().sort_values(by='mes_ano')
    
    fig_temporal = go.Figure()
    fig_temporal.add_trace(go.Scatter(
        x=g1_data['mes_ano'], y=g1_data['valor_total'], mode='lines+markers',
        line=dict(color=COR_DESTAQUE, width=3),
        marker=dict(size=7, color=COR_DESTAQUE, line=dict(color='#ffffff', width=1.5)),
        hovertemplate="<b>📅 Período:</b> %{x}<br><b>💵 Gasto:</b> R$ %{y:,.2f}<extra></extra>"
    ))
    fig_temporal.update_layout(
        template=TEMA_EXECUTIVO, height=H_GRAFICO, width=W_GRAFICO, margin=dict(l=55, r=35, t=55, b=25),
        title=dict(text="<b>Evolução Mensal do Custo Consolidado</b>", font=dict(size=12, color='#1e293b'), x=0.01, y=0.92),
        yaxis=dict(gridcolor="#f1f5f9", tickfont=dict(size=9, color="#475569")),
        xaxis=dict(showgrid=False, tickfont=dict(size=9, color="#475569"))
    )

    # =======================================================================
    # RANKING ÓRGÃO PAGADOR
    # =======================================================================
    fig_p7 = go.Figure()
    invalidos = ['None', 'NaN', 'null', '', 'SEM INFORMAÇÃO', 'NÃO INFORMADO', 'NÃO INFORMADA']
    df_p_v = df_p_f[~df_p_f['nome_orgao_pagador'].astype(str).str.strip().isin(invalidos)] if not df_p_f.empty else pd.DataFrame()
    
    if not df_p_v.empty:
        g7 = df_p_v.groupby('nome_orgao_pagador')['valor'].agg(['sum', 'count']).reset_index()
        g7_f = g7.sort_values(by='sum', ascending=False).head(4).sort_values(by='sum', ascending=True)
        
        if not g7_f.empty:
            cores_g7 = [COR_NEUTRA] * (len(g7_f) - 1) + [COR_DESTAQUE]
            nomes_curtos = [n[:22] + '...' if len(n) > 22 else n for n in g7_f['nome_orgao_pagador']]
            
            fig_p7.add_trace(go.Bar(
                y=nomes_curtos, x=g7_f['sum'], orientation='h',
                marker=dict(color=cores_g7, line=dict(color='#ffffff', width=1)),
                text=[f" <b>R$ {val/1e6:.1f}M</b> " for val in g7_f['sum']], textposition='outside',
                textfont=dict(color=COR_DESTAQUE, size=9),
                hovertemplate="<b>🏛️ Órgão Pagador:</b> %{customdata[0]}<br><b>💵 Total:</b> R$ %{x:,.2f}<br><b>📊 Ordens:</b> %{customdata[1]:,}<extra></extra>",
                customdata=list(zip(g7_f['nome_orgao_pagador'], g7_f['count'])), width=0.55
            ))
    fig_p7.update_layout(
        template=TEMA_EXECUTIVO, height=H_GRAFICO, width=W_GRAFICO, margin=dict(l=140, r=75, t=55, b=25), showlegend=False,
        title=dict(text="<b>Top Órgãos Pagadores por Desembolso Consolidado</b>", font=dict(size=12, color='#1e293b'), x=0.01, y=0.92),
        xaxis=dict(visible=False, range=[0, g7_f['sum'].max() * 1.35] if not df_p_v.empty and not g7_f.empty else [0, 1]),
        yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", tickfont=dict(size=9, color='#1e293b', weight="bold"))
    )

    # =======================================================================
    # DESTINOS COM MAIOR CUSTO MÉDIO
    # =======================================================================
    g2 = df_v_f.groupby('destino_estado')['valor_total'].agg(['mean', 'count']).reset_index()
    g2_f = g2[(g2['count'] >= 5) & (~g2['destino_estado'].isin(["Sigiloso", "Não Informado", "Não informado"]))].sort_values(by='mean', ascending=False).head(4).sort_values(by='mean', ascending=True)
    
    fig_p2 = go.Figure()
    if not g2_f.empty:
        cores_g2 = [COR_NEUTRA] * (len(g2_f) - 1) + [COR_DESTAQUE]
        fig_p2.add_trace(go.Bar(
            y=g2_f['destino_estado'], x=g2_f['mean'], orientation='h',
            marker=dict(color=cores_g2, line=dict(color='#ffffff', width=1)),
            text=[f" <b>R$ {val:,.2f}</b> " for val in g2_f['mean']], textposition='outside',
            textfont=dict(color=COR_DESTAQUE, size=9),
            hovertemplate="<b>📍 Destino:</b> %{y}<br><b>💵 Média:</b> R$ %{x:,.2f}<br><b>✈️ Viagens:</b> %{customdata}<extra></extra>",
            customdata=g2_f['count'], width=0.55
        ))
    fig_p2.update_layout(
        template=TEMA_EXECUTIVO, height=H_GRAFICO, width=W_GRAFICO, margin=dict(l=140, r=85, t=55, b=25), showlegend=False,
        title=dict(text="<b>Top Destinos por Maior Custo Médio por Viagem (N ≥ 5)</b>", font=dict(size=12, color='#1e293b'), x=0.01, y=0.92),
        xaxis=dict(visible=False, range=[0, g2_f['mean'].max() * 1.3] if not g2_f.empty else [0, 1]),
        yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", tickfont=dict(size=10, color='#1e293b', weight="bold"))
    )

    # =======================================================================
    # TIPO DE PAGAMENTO POR VALOR MÉDIO
    # =======================================================================
    fig_p4 = go.Figure()
    if not df_p_f.empty:
        g3 = df_p_f.groupby('tipo_pagamento')['valor'].agg(['mean', 'count']).reset_index().sort_values(by='mean', ascending=True)
        cores_g3 = [COR_NEUTRA] * (len(g3) - 1) + [COR_DESTAQUE]
        fig_p4.add_trace(go.Bar(
            y=g3['tipo_pagamento'], x=g3['mean'], orientation='h',
            marker=dict(color=cores_g3, line=dict(color='#ffffff', width=1)),
            text=[f" <b>R$ {val:,.2f}</b> " for val in g3['mean']], textposition='outside',
            textfont=dict(color=COR_DESTAQUE, size=9),
            hovertemplate="<b>💳 Tipo:</b> %{y}<br><b>💵 Média:</b> R$ %{x:,.2f}<br><b>📊 Lançamentos:</b> %{customdata:,}<extra></extra>",
            customdata=g3['count'], width=0.55
        ))
    fig_p4.update_layout(
        template=TEMA_EXECUTIVO, height=H_GRAFICO, width=W_GRAFICO, margin=dict(l=140, r=85, t=55, b=25), showlegend=False,
        title=dict(text="<b>Modalidades de Pagamento por Valor Médio Lançado</b>", font=dict(size=12, color='#1e293b'), x=0.01, y=0.92),
        xaxis=dict(visible=False, range=[0, g3['mean'].max() * 1.35] if not df_p_f.empty else [0, 1]),
        yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", tickfont=dict(size=10, color='#1e293b', weight="bold"))
    )

    # =======================================================================
    # MEIOS DE TRANSPORTE MAIS UTILIZADOS
    # =======================================================================
    fig_p5 = go.Figure()
    if not df_t_f.empty:
        df_t_v = df_t_f[~df_t_f['meio_transporte'].isin([None, 'Sem informação', ''])]
        g4 = df_t_v['meio_transporte'].value_counts().reset_index()
        g4.columns = ['meio_transporte', 'quantidade']
        g4_g = g4.head(4).sort_values(by='quantidade', ascending=True)
        
        if not g4_g.empty:
            cores_g4 = [COR_NEUTRA] * (len(g4_g) - 1) + [COR_DESTAQUE]
            fig_p5.add_trace(go.Bar(
                y=g4_g['meio_transporte'], x=g4_g['quantidade'], orientation='h',
                marker=dict(color=cores_g4, line=dict(color='#ffffff', width=1)),
                text=[f" <b>{val:,}</b> " for val in g4_g['quantidade']], textposition='outside',
                textfont=dict(color=COR_DESTAQUE, size=9),
                hovertemplate="<b>🚀 Modal:</b> %{y}<br><b>📊 Trechos:</b> %{x:,}<extra></extra>", width=0.55
            ))
    fig_p5.update_layout(
        template=TEMA_EXECUTIVO, height=H_GRAFICO, width=W_GRAFICO, margin=dict(l=140, r=65, t=55, b=25), showlegend=False,
        title=dict(text="<b>Volumetria Absoluta de Meios de Transporte</b>", font=dict(size=12, color='#1e293b'), x=0.01, y=0.92),
        xaxis=dict(visible=False, range=[0, g4_g['quantidade'].max() * 1.25] if not df_t_f.empty and not g4_g.empty else [0, 1]),
        yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", tickfont=dict(size=10, color='#1e293b', weight="bold"))
    )

    # =======================================================================
    # UFs DE DESTINO FREQUENTES
    # =======================================================================
    fig_p6 = go.Figure()
    if not df_t_f.empty:
        df_u_v = df_t_f[~df_t_f['destino_uf'].isin([None, 'Sem informação', ''])]
        if not df_u_v.empty:
            g5 = df_u_v['destino_uf'].value_counts().reset_index()
            g5.columns = ['uf_destino', 'total_trechos']
            g5_g = g5.head(4).sort_values(by='total_trechos', ascending=True)
            
            cores_g5 = [COR_NEUTRA] * (len(g5_g) - 1) + [COR_DESTAQUE]
            fig_p6.add_trace(go.Bar(
                y=g5_g['uf_destino'], x=g5_g['total_trechos'], orientation='h',
                marker=dict(color=cores_g5, line=dict(color='#ffffff', width=1)),
                text=[f" <b>{val:,}</b> " for val in g5_g['total_trechos']], textposition='outside',
                textfont=dict(color=COR_DESTAQUE, size=9),
                hovertemplate="<b>📍 UF Destino:</b> %{y}<br><b>📊 Total Trechos:</b> %{x:,}<extra></extra>", width=0.55
            ))
    fig_p6.update_layout(
        template=TEMA_EXECUTIVO, height=H_GRAFICO, width=W_GRAFICO, margin=dict(l=140, r=65, t=55, b=25), showlegend=False,
        title=dict(text="<b>Top UFs de Destino Mais Frequentes nos Trechos</b>", font=dict(size=12, color='#1e293b'), x=0.01, y=0.92),
        xaxis=dict(visible=False, range=[0, g5_g['total_trechos'].max() * 1.25] if not df_t_f.empty and not g5_g.empty else [0, 1]),
        yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", tickfont=dict(size=10, color='#1e293b', weight="bold"))
    )

    # Estilização uniforme das caixas de hover para fundo escuro corporativo
    for f in [fig_temporal, fig_p7, fig_p2, fig_p4, fig_p5, fig_p6]:
        f.update_traces(hoverlabel=dict(bgcolor="#1e293b", bordercolor="#e2e8f0", font=dict(color="#ffffff", size=11, family="Segoe UI, Arial"), align="left"))

    # Captura e renderização em containers com alinhamento centralizado aproximado
    out_g1, out_g2 = widgets.Output(), widgets.Output()
    out_g3, out_g4 = widgets.Output(), widgets.Output()
    out_g5, out_g6 = widgets.Output(), widgets.Output()

    with out_g1: fig_temporal.show()
    with out_g2: fig_p7.show()
    with out_g3: fig_p2.show()
    with out_g4: fig_p4.show()
    with out_g5: fig_p5.show()
    with out_g6: fig_p6.show()

    # Alinhamento no 'centro' com gap explícito de 20px para juntar os blocos
    row1 = widgets.HBox([out_g1, out_g2], layout=widgets.Layout(justify_content='center', gap='20px', width='100%'))
    row2 = widgets.HBox([out_g3, out_g4], layout=widgets.Layout(justify_content='center', gap='20px', width='100%'))
    row3 = widgets.HBox([out_g5, out_g6], layout=widgets.Layout(justify_content='center', gap='20px', width='100%'))
    
    display(widgets.VBox([row1, row2, row3]))

# Inicialização e Renderização do Filtro Dropdown Interativo
lista_orgaos = ["Todos"] + sorted(df_viagem['nome_orgao_superior'].dropna().unique().tolist())

interact(
    renderizar_dashboard_interativo, 
    orgao_selecionado=widgets.Dropdown(
        options=lista_orgaos, 
        value="Todos", 
        description="Filtrar Órgão:",
        style={'description_width': 'initial'},
        layout={'width': '450px', 'margin': '5px 0'}
    )
);

interactive(children=(Dropdown(description='Filtrar Órgão:', layout=Layout(margin='5px 0', width='450px'), opt…

---
### 🏛️ 4. Camada Gold Agregada (Persistência no PostgreSQL)

In [67]:
print("[*] Processando agregação para a camada Gold...")

df_gold_agregada = df_viagem.groupby('nome_orgao_superior').agg(
    total_viagens=('id_viagem', 'count'),
    custo_total=('valor_total', 'sum'),
    custo_medio=('valor_total', 'mean'),
    duracao_media_dias=('duracao_dias', 'mean')
).reset_index()

df_gold_agregada['custo_medio'] = df_gold_agregada['custo_medio'].round(2)
df_gold_agregada['duracao_media_dias'] = df_gold_agregada['duracao_media_dias'].round(1)

# Tratamento estrito de valores nulos matemáticos antes de persistir no Postgres
df_gold_agregada = df_gold_agregada.where(pd.notnull(df_gold_agregada), None)

conexao = conectar()

sql_ddl = """
DROP TABLE IF EXISTS gold_resumo_orgaos CASCADE;
CREATE TABLE gold_resumo_orgaos (
    nome_orgao_superior VARCHAR(255) PRIMARY KEY,
    total_viagens INT,
    custo_total DECIMAL(14,2),
    custo_medio DECIMAL(12,2),
    duracao_media_dias DECIMAL(5,1)
);
"""
executar(conexao, sql_ddl)

colunas = ['nome_orgao_superior', 'total_viagens', 'custo_total', 'custo_medio', 'duracao_media_dias']
placeholders = ", ".join(["%s"] * len(colunas))
sql_insert = f"INSERT INTO gold_resumo_orgaos ({', '.join(colunas)}) VALUES ({placeholders})"

registros = [tuple(x) for x in df_gold_agregada.itertuples(index=False, name=None)]
inserir_em_lote(conexao, sql_insert, registros)

# SALVAMENTO FÍSICO COM COMMIT
conexao.commit()
conexao.close()

print(f"[SUCCESS] Tabela 'gold_resumo_orgaos' gravada com sucesso!")
print(f"  • {len(df_gold_agregada)} órgãos persistidos na camada Gold.")

conexao = conectar()

sql_view = """
DROP VIEW IF EXISTS gold_resumo_orgaos_view;

CREATE VIEW gold_resumo_orgaos_view AS
WITH pagamentos_agrupados AS (
    -- Agrupa os pagamentos primeiro para garantir relação 1:1 no JOIN posterior
    SELECT 
        id_viagem, 
        COUNT(id_pagamento) AS total_lancamentos
    FROM silver_pagamento
    GROUP BY id_viagem
)
SELECT 
    v.nome_orgao_superior,
    COUNT(v.id_viagem) AS total_viagens,
    SUM(v.valor_total) AS custo_total,
    ROUND(AVG(v.valor_total)::numeric, 2) AS custo_medio,
    ROUND(AVG(v.duracao_dias)::numeric, 1) AS duracao_media_dias,
    SUM(COALESCE(p.total_lancamentos, 0)) AS total_lancamentos_pagamento
FROM silver_viagem v
LEFT JOIN pagamentos_agrupados p ON v.id_viagem = p.id_viagem
WHERE v.nome_orgao_superior IS NOT NULL AND v.nome_orgao_superior != ''
GROUP BY v.nome_orgao_superior;
"""

try:
    executar(conexao, sql_view)
    conexao.commit()
    print("[SUCCESS] VIEW Gold implementada!")
except Exception as e:
    conexao.rollback()
    print(f"[!] Erro ao refatorar VIEW: {e}")
finally:
    conexao.close()

[*] Processando agregação para a camada Gold...
[SUCCESS] Tabela 'gold_resumo_orgaos' gravada com sucesso!
  • 35 órgãos persistidos na camada Gold.
[SUCCESS] VIEW Gold implementada!


---
### 🔎 5. Validação e Consulta Direta da Camada Gold

In [68]:
print("[*] Conectando ao banco de dados para validar a tabela 'gold_resumo_orgaos'...")

conexao = conectar()
query_validacao = """
    SELECT 
        nome_orgao_superior, 
        total_viagens, 
        custo_total, 
        custo_medio, 
        duracao_media_dias
    FROM gold_resumo_orgaos
    ORDER BY custo_total DESC
    LIMIT 10;
"""
df_gold_verificacao = pd.read_sql_query(query_validacao, conexao)
conexao.close()

print(f"[+] Consulta de auditoria concluída com sucesso!")
print(f"  • Exibindo os 10 órgãos de maior relevância orçamentária persistidos na base:")
display(df_gold_verificacao)

[*] Conectando ao banco de dados para validar a tabela 'gold_resumo_orgaos'...
[+] Consulta de auditoria concluída com sucesso!
  • Exibindo os 10 órgãos de maior relevância orçamentária persistidos na base:


,nome_orgao_superior,total_viagens,custo_total,custo_medio,duracao_media_dias
0,Ministério da Justiça e Segurança Pública,75742,4.869331e+08,6428.84,17.1
1,Ministério da Defesa,61912,1.560703e+08,2520.84,5.2
2,Ministério da Educação,65295,1.112913e+08,1704.44,2.7
3,Ministério do Meio Ambiente e Mudança do Clima,19413,4.969771e+07,2560.02,7.1
4,Ministério da Previdência Social,8190,4.041731e+07,4934.96,12.1
5,Ministério da Saúde,8672,3.904983e+07,4502.98,4.2
6,Ministério da Fazenda,11854,3.247439e+07,2739.53,3.7
7,Ministério dos Povos Indígenas,5101,2.628692e+07,5153.29,11.1
8,Ministério das Relações Exteriores,2014,2.553355e+07,12678.03,9.4
9,Ministério do Desenvolvimento Agrário e Agricu...,8581,2.199842e+07,2563.62,4.3


---
### 🔎 6. Três Novas Perguntas de Negócio e Gráficos Diretos sobre a VIEW Gold

In [74]:
conexao = conectar()

# Configurações de estilo unificadas para reutilização nos hovers
HOVER_STYLE = dict(
    bgcolor="#1e293b",
    bordercolor="#e2e8f0",
    font=dict(color="#ffffff", size=12, family="Segoe UI, Arial"),
    align="left"
)

# =======================================================================
# QUESTÃO GOLD 1 (extra): Quais os 5 órgãos com maior duração média de viagens?
# =======================================================================
query_g1 = """
    SELECT nome_orgao_superior, duracao_media_dias, total_viagens
    FROM gold_resumo_orgaos_view
    ORDER BY duracao_media_dias DESC
    LIMIT 5;
"""
df_gold_p1 = pd.read_sql_query(query_g1, conexao)
df_gold_p1.index = df_gold_p1.index + 1
print("\n=== Tabela Gold 1: Maior Duração Média ===")
display(df_gold_p1)

# Inverte o dataframe para manter o maior valor no topo do gráfico horizontal
df_g1_grafico = df_gold_p1.sort_values(by='duracao_media_dias', ascending=True)
cores_g1 = ['#cbd5e1'] * (len(df_g1_grafico) - 1) + ['#1e3a8a']  # Destaque Azul Marinho no líder

fig_g1 = go.Figure()
fig_g1.add_trace(go.Bar(
    y=df_g1_grafico['nome_orgao_superior'],
    x=df_g1_grafico['duracao_media_dias'],
    orientation='h',
    marker=dict(color=cores_g1, line=dict(color='#ffffff', width=1.5)),
    text=[f" <b>{val:.1f} dias</b> " for val in df_g1_grafico['duracao_media_dias']],
    textposition='outside',
    textfont=dict(color='#1e293b', size=11, family="Segoe UI, Arial"),
    hovertemplate=(
        "<b>🏛️ Órão:</b> %{y}<br>"
        "<b>⏱️ Duração Média:</b> %{x:.1f} dias<br>"
        "<b>✈️ Total Viagens:</b> %{customdata:,}<extra></extra>"
    ),
    customdata=df_g1_grafico['total_viagens'],
    width=0.55
))

fig_g1.update_traces(hoverlabel=HOVER_STYLE)
fig_g1.update_layout(
    title=dict(
        text="<b>Top 5 Órgãos por Maior Duração Média das Rotas</b><br><span style='font-size: 11px; color: #64748b;'>Métricas extraídas via VIEW agregada na Camada Gold</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"), x=0.02, y=0.92
    ),
    template="plotly_white", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff", height=380,
    margin=dict(l=320, r=120, t=90, b=30), showlegend=False,
    xaxis=dict(visible=False, range=[0, df_g1_grafico['duracao_media_dias'].max() * 1.25]),
    yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", linewidth=1.5, tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold"))
)
fig_g1.show()


# =======================================================================
# QUESTÃO GOLD 2 (extra): Quais órgãos possuem custo médio por viagem crítico (> R$ 4.000)?
# =======================================================================
query_g2 = """
    SELECT nome_orgao_superior, custo_medio, total_viagens
    FROM gold_resumo_orgaos_view
    WHERE custo_medio > 4000.00
    ORDER BY custo_medio DESC;
"""
df_gold_p2 = pd.read_sql_query(query_g2, conexao)
df_gold_p2.index = df_gold_p2.index + 1
print("\n=== Tabela Gold 2: Custo Médio Crítico (> R$ 4k) ===")
display(df_gold_p2)

df_g2_grafico = df_gold_p2.sort_values(by='custo_medio', ascending=True)
cores_g2 = ['#cbd5e1'] * (len(df_g2_grafico) - 1) + ['#b45309']  # Destaque Ouro Velho/Âmbar para custo crítico

fig_g2 = go.Figure()
fig_g2.add_trace(go.Bar(
    y=df_g2_grafico['nome_orgao_superior'],
    x=df_g2_grafico['custo_medio'],
    orientation='h',
    marker=dict(color=cores_g2, line=dict(color='#ffffff', width=1.5)),
    text=[f" <b>R$ {val:,.2f}</b> " for val in df_g2_grafico['custo_medio']],
    textposition='outside',
    textfont=dict(color='#b45309', size=11, family="Segoe UI, Arial"),
    hovertemplate=(
        "<b>🏛️ Órgão:</b> %{y}<br>"
        "<b>💵 Custo Médio por Rota:</b> R$ %{x:,.2f}<br>"
        "<b>✈️ Total Viagens:</b> %{customdata:,}<extra></extra>"
    ),
    customdata=df_g2_grafico['total_viagens'],
    width=0.55
))

fig_g2.update_traces(hoverlabel=HOVER_STYLE)
fig_g2.update_layout(
    title=dict(
        text="<b>Órgãos com Custo Médio por Viagem Superior a R$ 4.000,00</b><br><span style='font-size: 11px; color: #64748b;'>Indicador de criticidade financeira mapeado diretamente da Camada Gold</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"), x=0.02, y=0.92
    ),
    template="plotly_white", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff", height=380,
    margin=dict(l=320, r=120, t=90, b=30), showlegend=False,
    xaxis=dict(visible=False, range=[0, df_g2_grafico['custo_medio'].max() * 1.25]),
    yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", linewidth=1.5, tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold"))
)
fig_g2.show()


# =======================================================================
# QUESTÃO GOLD 3 (extra): Qual a correlação volumétrica entre total de viagens e custo total?
# =======================================================================
query_g3 = """
    SELECT total_viagens, custo_total, nome_orgao_superior
    FROM gold_resumo_orgaos_view
    ORDER BY custo_total DESC;
"""
df_gold_p3 = pd.read_sql_query(query_g3, conexao)
df_gold_p3.index = df_gold_p3.index + 1
print("\n=== Tabela Gold 3: Dispersão Volumétrica vs Desembolso ===")
display(df_gold_p3.head(5))

fig_g3 = go.Figure()
fig_g3.add_trace(go.Scatter(
    x=df_gold_p3['total_viagens'],
    y=df_gold_p3['custo_total'],
    mode='markers',
    marker=dict(
        size=12,
        color='#0f766e',  # Paleta Teal Corporativo
        line=dict(color='#ffffff', width=1.5),
        opacity=0.85
    ),
    hovertemplate=(
        "<b>🏛️ Órgão Superior:</b> %{customdata}<br>"
        "<b>✈️ Volume de Viagens:</b> %{x:,}<br>"
        "<b>💵 Desembolso Total:</b> R$ %{y:,.2f}<extra></extra>"
    ),
    customdata=df_gold_p3['nome_orgao_superior']
))

fig_g3.update_traces(hoverlabel=HOVER_STYLE)
fig_g3.update_layout(
    title=dict(
        text="<b>Matriz de Dispersão Orçamentária: Volumetria vs Desembolso</b><br><span style='font-size: 11px; color: #64748b;'>Análise macro de correlação linear e detecção de clusters entre órgãos</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"), x=0.02, y=0.92
    ),
    template="plotly_white", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff", height=450,
    margin=dict(l=80, r=40, t=90, b=60), showlegend=False,
    xaxis=dict(
        title=dict(text="Volume Absoluto de Viagens", font=dict(size=11, color='#475569')),
        gridcolor="#f1f5f9", showgrid=True, showline=True, linecolor="#e2e8f0"
    ),
    yaxis=dict(
        title=dict(text="Custo Total Acumulado (R$)", font=dict(size=11, color='#475569')),
        gridcolor="#f1f5f9", showgrid=True, showline=True, linecolor="#e2e8f0"
    )
)
fig_g3.show()

# =======================================================================
# QUESTÃO GOLD 4 (extra): Qual ministério apresenta a maior complexidade transacional (maior média de lançamentos de pagamento gerados por viagem)??
# =======================================================================
query_complexidade = """
    SELECT 
        nome_orgao_superior,
        total_viagens,
        total_lancamentos_pagamento,
        ROUND((total_lancamentos_pagamento::numeric / total_viagens), 2) AS pagamentos_por_viagem
    FROM gold_resumo_orgaos_view
    WHERE total_viagens >= 100 
    ORDER BY pagamentos_por_viagem DESC
    LIMIT 5;
"""

df_complexidade = pd.read_sql_query(query_complexidade, conexao)
df_complexidade.index = df_complexidade.index + 1
print("\n=== Tabela Gold: Adensamento de Lançamentos Financeiros por Viagem ===")
display(df_complexidade)

# Ordenação para posicionar o maior índice no topo do gráfico horizontal
df_comp_grafico = df_complexidade.sort_values(by='pagamentos_por_viagem', ascending=True)
cores_g4 = ['#cbd5e1'] * (len(df_comp_grafico) - 1) + ['#1e3a8a']  

fig_g4 = go.Figure()
fig_g4.add_trace(go.Bar(
    y=df_comp_grafico['nome_orgao_superior'],
    x=df_comp_grafico['pagamentos_por_viagem'],
    orientation='h',
    marker=dict(color=cores_g4, line=dict(color='#ffffff', width=1.5)),
    text=[f" <b>{val:.2f} pag/vj</b> " for val in df_comp_grafico['pagamentos_por_viagem']],
    textposition='outside',
    textfont=dict(color='#1e293b', size=11, family="Segoe UI, Arial"),
    hovertemplate=(
        "<b>🏛️ Órgão:</b> %{y}<br>"
        "<b>💳 Média de Lançamentos:</b> %{x:.2f} eventos por viagem<br>"
        "<b>✈️ Total Viagens:</b> %{customdata[0]:,}<br>"
        "<b>💵 Total Lançamentos:</b> %{customdata[1]:,}<extra></extra>"
    ),
    customdata=list(zip(df_comp_grafico['total_viagens'], df_comp_grafico['total_lancamentos_pagamento'])),
    width=0.55
))

fig_g4.update_traces(hoverlabel=HOVER_STYLE)
fig_g4.update_layout(
    title=dict(
        text="<b>Top 5 Órgãos por Complexidade Transacional (Média de Pagamentos por Viagem)</b><br><span style='font-size: 11px; color: #64748b;'>Análise de desdobramentos de despesas (diárias, passagens, taxas) por missão iniciada</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"), x=0.02, y=0.92
    ),
    template="plotly_white", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff", height=380,
    margin=dict(l=320, r=120, t=90, b=30), showlegend=False,
    xaxis=dict(visible=False, range=[0, df_comp_grafico['pagamentos_por_viagem'].max() * 1.3]),
    yaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", linewidth=1.5, tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold"))
)
fig_g4.show()

# Encerramento seguro da sessão
conexao.close()


=== Tabela Gold 1: Maior Duração Média ===


,nome_orgao_superior,duracao_media_dias,total_viagens
1,Ministério da Justiça e Segurança Pública,17.1,75742
2,Ministério da Previdência Social,12.1,8190
3,Ministério dos Povos Indígenas,11.1,5101
4,Ministério das Relações Exteriores,9.4,2014
5,Ministério do Meio Ambiente e Mudança do Clima,7.1,19413



=== Tabela Gold 2: Custo Médio Crítico (> R$ 4k) ===


,nome_orgao_superior,custo_medio,total_viagens
1,Ministério das Relações Exteriores,12678.03,2014
2,Ministério do Turismo,10546.60,337
3,Ministério do Esporte,7228.99,245
4,Banco Central do Brasil - Orçamento Fiscal e S...,6792.27,385
5,Ministério da Justiça e Segurança Pública,6428.84,75742
6,Ministério das Mulheres,6181.07,462
7,Ministério das Comunicações,5838.63,1770
8,Ministério da Igualdade Racial,5796.16,508
9,"Ministério do Desenvolvimento, Indústria, Comé...",5255.41,1684
10,Ministério dos Povos Indígenas,5153.29,5101



=== Tabela Gold 3: Dispersão Volumétrica vs Desembolso ===


,total_viagens,custo_total,nome_orgao_superior
1,75742,4.869331e+08,Ministério da Justiça e Segurança Pública
2,61912,1.560703e+08,Ministério da Defesa
3,65295,1.112913e+08,Ministério da Educação
4,19413,4.969771e+07,Ministério do Meio Ambiente e Mudança do Clima
5,8190,4.041731e+07,Ministério da Previdência Social



=== Tabela Gold: Adensamento de Lançamentos Financeiros por Viagem ===


,nome_orgao_superior,total_viagens,total_lancamentos_pagamento,pagamentos_por_viagem
1,Ministério do Turismo,337,1163.0,3.45
2,Ministério da Igualdade Racial,508,1512.0,2.98
3,"Ministério do Empreendedorismo, da Microempres...",272,767.0,2.82
4,Ministério das Mulheres,462,1289.0,2.79
5,Ministério das Cidades,581,1607.0,2.77
